In [ ]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import os
import asyncio
import requests
from agents import Agent, Runner, trace, function_tool, FunctionTool
import gradio as gr
from pydantic import BaseModel


In [ ]:

load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# For pushover
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
@function_tool
def push(input: str) -> str:
    """Send out a push notification message"""
    show(input)
    if not input:
        return "No message provided"
    
    payload = {"user": pushover_user, "token": pushover_token, "message": input}
    response = requests.post(pushover_url, data=payload)
    
    if response.status_code == 200:
        return "Push notification sent: " + input
    else:
        return f"Push notification failed with status {response.status_code}: {response.text}"

In [ ]:
# Initialize Groceries dicts
groceries_inventory = {"eggs": 12, "milk": 16, "bread": 10, "bananas": 6}
groceries_consumed = {}
richConsole = True

In [ ]:
groceries_inventory

In [ ]:
def _mark_complete(item: str) -> str:
    """Internal helper for mark_complete logic."""
    result = ""
    if item in groceries_inventory.keys():
        quantity = groceries_inventory[item]
        if quantity == 0:
            result += f"[red]Please order {item} at this time[/red]\n" if richConsole else f"Please order {item} at this time\n"
    else:
        print("Item not found in the grocery list")
    return result

In [ ]:
@function_tool
def mark_complete(item: str) -> str:
    """
    Marks the grocery item as fully used and hence it is time to reorder. Handoffs to the push tool to send a notification.
    Args:
        item: A string containing the item name.
    """
    return _mark_complete(item)

In [ ]:
def update_groceries_status() -> str:
    """
    Updates the status of the groceries items in the format of item: quantity. If the quantity is zero, it calls mark_complete.
    Args:
        None
    """
    remaining = {
        key: groceries_inventory[key] - groceries_consumed.get(key, 0)
        for key in groceries_inventory
    }

    result = ""
    for grocery, quantity in remaining.items():
        if quantity == 0:
            result += _mark_complete(grocery)  # call helper directly
            result += f"Grocery #{grocery}: [red]{quantity}[/red]\n" if richConsole else f"Grocery #{grocery}: {quantity}\n"
        else:
            result += f"Grocery #{grocery}: {quantity}\n"

    groceries_inventory.update(remaining)
    for key in remaining:
        groceries_consumed[key] = 0

    if richConsole:
        show(result)

    return result

In [ ]:
class ConsumedItems(BaseModel):
    items: list[str]
    quantities: list[int]

In [ ]:
@function_tool
def update_groceries_consumed(consumed: ConsumedItems) -> str:
    """
    Updates the inventory by recording consumed items.
    Args:
        consumed: An object with parallel lists of item names and quantities.
    """
    result_lines = []
    
    for grocery, quantity in zip(consumed.items, consumed.quantities):
        if grocery not in groceries_inventory:
            result_lines.append(f"Item '{grocery}' not found in the grocery list.")
            continue
        
        if quantity < 0:
            result_lines.append(f"Invalid quantity for {grocery}: {quantity}")
            continue
            
        available = groceries_inventory[grocery] - groceries_consumed.get(grocery, 0)
        if quantity > available:
            result_lines.append(f"You cannot consume more {grocery} than available ({available}).")
            continue
            
        groceries_consumed[grocery] = groceries_consumed.get(grocery, 0) + quantity
        result_lines.append(f"Successfully updated {grocery}.")

    final_report = "\n".join(result_lines)
    
    if richConsole:
        print(final_report) 
        
    return final_report

In [ ]:
class GroceryItems(BaseModel):
    names: list[str]
    quantities: list[int]

In [ ]:
@function_tool
def create_groceries_inventory(inventory: GroceryItems) -> str:
    """Creates the groceries inventory from a list of GroceryItems."""
    for item, quantity in zip(inventory.names, inventory.quantities):
        groceries_inventory[item] = quantity
    result = update_groceries_status()
    return result

In [ ]:
tools = [push]
instructions ="You push and send notifications. You receive a message to be sent and you push that using the tool provided. Use only the tool provided"


push_agent = Agent(
    name="Push Agent",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Push a notification message")

In [ ]:
tools = [create_groceries_inventory, mark_complete, update_groceries_consumed]
handoffs = [push_agent]

In [ ]:
instructions = """
You are a smart refrigerator assistant. You manage the user's grocery inventory and consumption.

**Interpreting the user:** Parse item names and quantities from natural or shorthand phrasing. All of these mean the same thing and should be interpreted as one item with a quantity:
- "I have 10 eggs" / "10 eggs" / "eggs 10" / "eggs: 10" → eggs: 10
- "I used 2 milk" / "milk 2" / "2 cups milk" → milk: 2 (use the number given; treat "cups" or "loaves" as the unit, quantity is the number)
Normalize item names to simple lowercase words (e.g. eggs, milk, bread, bananas) for tool calls. If the user says "I have 10 eggs" when setting up, pass {"eggs": 10}; if they say "eggs 10" when reporting consumption, pass {"eggs": 10} to update_groceries_consumed.

**Setup:** When the user gives a starting inventory (in any of the phrasings above), use the create_groceries_inventory tool to map item names to quantities (integers).

**When the user reports consumption:** Use the update_groceries_consumed tool with item names to quantities consumed (interpret "I used X", "X 3", "3 X", etc. as that item and quantity). Then use the update_groceries_status tool to update status of the inventory

**When any item's remaining quantity is zero:** Use the mark_complete tool for that item so the user is notified to reorder it. Then handoff to the push tool to notify the user

**Rules:** Use only the tools above; do not invent tools. If the user does not specify a quantity, infer a reasonable amount or use 0. Reply in Rich console markup where helpful; no code blocks only if the richConsole is set to True. Do not ask for clarification—use your tools and then respond with a clear summary or status.
"""

In [ ]:

smart_refrigerator = Agent(name="Smart Refigerator", instructions=instructions, tools=tools, handoffs=handoffs, model="gpt-4o-mini")

message = "I'll tell you when I use or consume some of the groceries. Please tell me when I need to reorder any item (i.e. when something runs out)"

In [ ]:
async def chat(message, history):
    # Strip any extra fields (like 'metadata') that Gradio adds
    clean_history = [{"role": m["role"], "content": m["content"]} for m in history]
    messages = clean_history + [{"role": "user", "content": message}]
    
    with trace("Groceries Inventory"):
        result = await Runner.run(smart_refrigerator, messages)
        return result.final_output

In [ ]:
richConsole = False
groceries_inventory = {}
groceries_consumed = {}
gr.ChatInterface(
    fn=chat,
    title="Smart Refrigerator Agent",
    type="messages",
    save_history=True,
    description="Ask about your groceries inventory, and the agent will respond."
).launch()